# Train YOLOv8-OBB on Synthetic QR Code Dataset

**Instructions:**
1.  Run this notebook (Runtime -> Run all).
2.  **Ensure** the `generated_obb_dataset` folder exists in the same directory as this notebook.
3.  The training will start automatically.
4.  When finished, the best model (`best.pt`) will be available in `runs/obb/qr_obb_model/weights/`.

In [ ]:
# 1. Setup Environment
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

In [ ]:
# 2. Setup Data (Use Local Folder)
import os
import yaml

# Point to the existing local directory
DATASET_DIR = os.path.abspath('generated_obb_dataset')

if not os.path.exists(DATASET_DIR):
    print(f"Error: Dataset not found at {DATASET_DIR}")
else:
    print(f"Using dataset at: {DATASET_DIR}")

# 2.3 Create dataset.yaml
# Note: We verify the structure matches YOLO requirements (images/train, labels/train, etc)
data_config = {
    'path': DATASET_DIR,
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/val', 
    'names': { 0: 'qr_code' }
}

with open('dataset.yaml', 'w') as f:
    yaml.dump(data_config, f)
print("dataset.yaml created.")

In [ ]:
# 3. Train YOLOv8-OBB
from ultralytics import YOLO

# Load the base OBB model
model = YOLO('yolov8n-obb.pt')

# Train
print("Starting training...")
results = model.train(
    data='dataset.yaml',
    epochs=15,             # 15 epochs is enough for this synthetic task
    imgsz=640,
    batch=16,
    name='qr_obb_model',
    exist_ok=True
)

In [ ]:
# 4. Validate & Download (Optional)
from IPython.display import Image

# Show sample prediction (check a file that actually exists)
validation_dir = os.path.join(DATASET_DIR, 'images/val')
if os.path.exists(validation_dir) and len(os.listdir(validation_dir)) > 0:
    sample_img = os.path.join(validation_dir, os.listdir(validation_dir)[0])
    print(f"Running inference on sample: {sample_img}")
    model.predict(sample_img, save=True, imgsz=640)
else:
    print("No validation images found to test.")

# Note: File download logic removed as we are running locally/assumed unpacked.
# The files are already on the system.